# Runge 函數插值分析
## 目標：找到達到 $10^{-10}$ 誤差所需的最小節點數

In [ ]:
import numpy as np
from scipy.interpolate import CubicSpline, barycentric_interpolate
import matplotlib.pyplot as plt
import time

# 設置中文字體（可選）
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
def runge_function(x):
    """Runge函數: f(x) = 1/(1 + 25x²)"""
    return 1 / (1 + 25 * x**2)

def compute_max_error(interp_func, true_func, x_test):
    """計算最大絕對誤差"""
    return np.max(np.abs(interp_func(x_test) - true_func(x_test)))

In [ ]:
# 尋找三次樣條插值的最小節點數
target_error = 1e-10
x_test = np.linspace(-1, 1, 10000)

print("尋找三次樣條插值的最小節點數...")
for n in range(100, 1001, 100):
    x_nodes = np.linspace(-1, 1, n)
    y_nodes = runge_function(x_nodes)
    cs = CubicSpline(x_nodes, y_nodes)
    error = compute_max_error(cs, runge_function, x_test)
    
    print(f"N = {n}, Error = {error:.2e}")
    
    if error < target_error:
        n_cubic = n
        error_cubic = error
        print(f"✓ 找到滿足條件的N: {n}, 誤差: {error:.2e}")
        break
else:
    n_cubic = 1000
    error_cubic = error
    print("在N ≤ 1000範圍內未找到滿足條件的解")

In [ ]:
# 尋找切比雪夫節點插值的最小節點數
print("\n尋找切比雪夫節點插值的最小節點數...")
for n in range(20, 101, 10):
    x_nodes = np.cos(np.linspace(0, np.pi, n))
    y_nodes = runge_function(x_nodes)
    
    def chebyshev_interpolator(x):
        return barycentric_interpolate(x_nodes, y_nodes, x)
    
    error = compute_max_error(chebyshev_interpolator, runge_function, x_test)
    
    print(f"N = {n}, Error = {error:.2e}")
    
    if error < target_error:
        n_chebyshev = n
        error_cheb = error
        print(f"✓ 找到滿足條件的N: {n}, 誤差: {error:.2e}")
        break
else:
    n_chebyshev = 100
    error_cheb = error
    print("在N ≤ 100範圍內未找到滿足條件的解")

In [ ]:
# 顯示最終結果
print("\n" + "=" * 60)
print("最終結果:")
print("=" * 60)
print(f"{'方法':<20} {'最小節點數':<12} {'最大誤差':<15}")
print(f"{'三次樣條插值':<20} {n_cubic:<12} {error_cubic:.2e}")
print(f"{'切比雪夫插值':<20} {n_chebyshev:<12} {error_cheb:.2e}")

In [ ]:
# 繪製比較圖形
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

x_eval = np.linspace(-1, 1, 1000)
y_true = runge_function(x_eval)

# 1. 原始函數
ax1.plot(x_eval, y_true, 'k-', linewidth=2)
ax1.set_title('Runge Function: $f(x) = \\frac{1}{1 + 25x^2}$')
ax1.grid(True, alpha=0.3)

# 2. 三次樣條插值
x_nodes_cubic = np.linspace(-1, 1, n_cubic)
y_nodes_cubic = runge_function(x_nodes_cubic)
cs = CubicSpline(x_nodes_cubic, y_nodes_cubic)
y_cubic = cs(x_eval)

ax2.plot(x_eval, y_true, 'k-', label='True')
ax2.plot(x_eval, y_cubic, 'r--', label='Cubic Spline')
ax2.scatter(x_nodes_cubic, y_nodes_cubic, color='red', s=20, label=f'Nodes (N={n_cubic})')
ax2.set_title('Cubic Spline Interpolation')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. 切比雪夫插值
x_nodes_cheb = np.cos(np.linspace(0, np.pi, n_chebyshev))
y_nodes_cheb = runge_function(x_nodes_cheb)
y_cheb = barycentric_interpolate(x_nodes_cheb, y_nodes_cheb, x_eval)

ax3.plot(x_eval, y_true, 'k-', label='True')
ax3.plot(x_eval, y_cheb, 'b--', label='Chebyshev')
ax3.scatter(x_nodes_cheb, y_nodes_cheb, color='blue', s=20, label=f'Nodes (N={n_chebyshev})')
ax3.set_title('Chebyshev Nodes Interpolation')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. 誤差比較
error_cubic_plot = np.abs(y_cubic - y_true)
error_cheb_plot = np.abs(y_cheb - y_true)

ax4.semilogy(x_eval, error_cubic_plot, 'r-', label='Cubic Spline Error')
ax4.semilogy(x_eval, error_cheb_plot, 'b-', label='Chebyshev Error')
ax4.axhline(y=1e-10, color='gray', linestyle='--', label='Target Error')
ax4.set_title('Interpolation Errors (Log Scale)')
ax4.set_ylabel('Absolute Error')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()